# Day 2 — You Are the QA Engineer

**Module 6 · Agentic RAG Testing**

---

Day 1 built the agent. Today you test it.

Not "run a script and see if the output looks right." **Systematic testing** — the kind that gives WidgetCo's engineering team something they can ship against. You'll start from three real incidents their support team already logged, design a test case for each one from scratch, write an LLM judge that makes the verdict reproducible, and run the whole suite against the live agent.

By the end you will have built:

| What | Starting from |
|---|---|
| 3 test scenarios grounded in real incidents | Support tickets WidgetCo's team actually filed |
| A golden dataset entry + hard negative for each | The specific wrong answer the agent has been giving |
| An LLM judge that makes each verdict reproducible | A prompt that encodes what "correct" means for this failure mode |
| An extended coverage matrix | The same one from Module 4 Day 4, with today's failure modes added |

**Estimated time:** 60 minutes
**Run order:** top to bottom. Cells that call the live agent need `.env` configured in this `examples/` folder (see Day 1's setup note). The golden-design and judge-definition cells run without credentials.

---

> **Where we are in the course**
> Day 1 built a working, traced 2-hop loop and named 4 new failure modes that only exist in multi-step retrieval.
> Module 5 Day 2 taught boundary value analysis on chunk size; Module 5 Day 3 taught hard negatives for faithfulness.
> Today both techniques come back — pointed at the planner's memory and its reasoning across hops instead of a single retrieval. And you're not reading about it. You're doing it.

---

## Your brief

**WidgetCo** is a B2B SaaS company founded in 2019. They build project-management and analytics tools for engineering teams — think sprint tracking, velocity dashboards, incident retrospectives. They're not a huge company. About 120 people, bootstrapped until a Series A in 2021, and still small enough that the Head of Support knows every enterprise account by name.

Their product line has evolved:

- **WidgetPro 2000** launched at founding. Solid product, but they sunset it in 2023 and replaced it with WidgetPro 3000. The 2000 had a `$50` cancellation fee; the 3000 dropped it entirely — cancel anytime, `$0`.
- **TurboMax** is their analytics add-on. It was called TurboMax 5 until a 2024 rebrand renamed it TurboMax Pro. Same `$299/year` price, new name.
- **Headquarters** moved from Austin to Denver in 2022 when they expanded the engineering team. The Denver office is lean — no walk-in support counter, everything handled online.

In 2023 they launched an AI support agent to handle the first line of customer queries — product questions, pricing, policies. By 2024 it was handling ~500 queries a day. Nobody tested it systematically when it shipped. It just worked well enough.

Then, three weeks ago, the Head of Support sent this:

> *"I need to flag something before our marketing campaign goes live. We've had three incidents from the AI agent that I can't explain. A customer migrating from WidgetPro 2000 was told the cancellation fee is $50 — it should be $0 on their new product. Another customer drove to the Denver office expecting to walk in and found a locked building. And a TurboMax Pro customer is disputing a $25 cancellation charge — I've checked every system, we never billed them $25, and that policy doesn't exist. We're about to 3× query volume. I need these fixed and tested before we flip the switch.*
>
> *Three incidents. Three different failure types. I can send you the exact queries if you need them."*

That last line — *three different failure types* — is the job. You're not here to fix the agent. You're here to write tests that catch these failures before the next release, on every future version of the agent.

Here's what you know going in:

| Incident | What the customer asked | What the agent said | What should have happened |
|---|---|---|---|
| 1 | Cancellation fee for the replacement for WidgetPro 2000 | `$50` | `$0` — that's WidgetPro 3000's fee |
| 2 | Same question, different run | `$0` (no product named) | `$0 for WidgetPro 3000` — product must be named |
| 3 | Cancellation fee for TurboMax Pro | `$25` | Admit the gap — no such policy exists |

By end of this notebook, each incident has a golden dataset entry, a hard negative that would have caught it, and an LLM judge that makes the verdict reproducible on every future agent version.

---

## Step 1 — Understand why your existing tools won't catch these incidents

Before writing a single test, you need to know which tools you already have and exactly where they break down. WidgetCo already has RAGAS from Module 5. You run it on the three incidents. Here's what happens:

**Incident 1 (wrong fee):** The agent's answer is *"The cancellation fee for the product that replaced WidgetPro 2000 is $50."* RAGAS `faithfulness` scores this **high** — `$50` is grounded in the retrieved context (it really was WidgetPro 2000's fee). `answer_relevancy` scores this **high** — the answer addresses the question. Both metrics pass. The customer is still wrong.

**Incident 2 (dropped identity):** The agent answers *"The cancellation fee is $0."* RAGAS scores this fine. The fee is correct, the answer is relevant. But which product? Nobody wrote a metric that checks what made it out of the retrieval loop into the final sentence.

**Incident 3 (fabricated fee):** The agent answers *"The cancellation fee for TurboMax Pro is $25."* RAGAS `faithfulness` flags this — `$25` doesn't appear in any retrieved chunk. But here's the problem: the agent's retrieval succeeded (it found the subscription price). If faithfulness sees *any* retrieved context, it may partially pass even on a fabricated answer, depending on how strictly the scorer is calibrated.

None of this is a flaw in RAGAS. These metrics were built for a system that retrieves exactly once:

- **`faithfulness`** asks "is every claim backed by some retrieved chunk?" — it cannot ask "was that the *right* chunk for this *entity*"
- **`answer_relevancy`** asks "does the response address the question?" — a confident wrong answer scores the same as a confident right one
- **`context_precision` / `context_recall`** score the *retrieval*, not whether the generator used it correctly

The failure modes in incidents 1–3 only become visible once you look at the *combination* of facts, not the individual facts. That's what you're building today: checks that look at what the agent did with what it retrieved.

> **The rule:** if an existing metric would catch an incident, you don't need a new one. If it wouldn't — even when you run it correctly — you do. All three incidents above need new checks.

---

## Step 2 — How to design a test case from an incident

Every test case in this notebook was designed using the same five-step process. Walk through it once before you see the first scenario — then you'll recognise it in each one.

**Step 1: Name the failure mode.**
Map the incident to one of the failure modes Day 1 defined. Is this premature_stop (the planner quit too early)? reasoning_chain_break (right facts, wrong combination or dropped identity)? ungraceful_failure (fabricated when it should have hedged)? Naming it before you write the test keeps you from writing a check that tests the wrong thing.

**Step 2: Write the question a real customer would ask.**
Not a synthetic benchmark question. The question that produced the incident. For incident 1, that's *"What is the cancellation fee for the product that replaced WidgetPro 2000?"* — because that's what a migrating customer actually asks.

**Step 3: Trace the hops the question requires.**
Map the question to the knowledge base facts it needs. How many retrievals does it take? Which fact comes first, which depends on which? This becomes your equivalence partition: is this a 1-hop question or a 2-hop question? A test suite with only 1-hop questions misses every multi-hop failure mode.

**Step 4: Write the golden entry.**
The golden is the *correct* answer and the conditions that define it. For reasoning-level checks, that means:
- `must_include`: facts the answer must mention (e.g. `["WidgetPro 3000", "$0"]`)
- `must_not_include`: facts that signal the wrong combination (e.g. `["$50"]`)
- `reference`: the ground-truth answer in plain English

**Step 5: Write the hard negative.**
The hard negative is the specific wrong answer you're testing for — the one that actually occurred in the incident. Not a random wrong answer. The *exact* failure mode. For incident 1, that's `"The cancellation fee for the product that replaced WidgetPro 2000 is $50."` — because that's what customers received.

**Step 6: Write the judge.**
The judge is an LLM prompt that takes the answer and returns `passed/score/reasoning`. The prompt must encode what "correct" means for this specific failure mode — not a generic "is this a good answer?" but "did the agent apply the RIGHT product's fee, not the discontinued one's?"

Every scenario below follows this sequence. Read the incident, watch the design decisions, then see the judge catch the hard negative.

---

## Scenario A — "The Memory Drop"
### Incident 2 · Failure mode: memory validation across hops

A WidgetPro 2000 customer was migrating their team to WidgetPro 3000. Before completing the move they asked the agent: *"What is the cancellation fee for the product that replaced WidgetPro 2000?"*

Here is exactly what the agent did:

```
Hop 1  query: "What replaced WidgetPro 2000?"
       found: "WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000."
       result: replacement = WidgetPro 3000  ✓

Hop 2  query: "What is WidgetPro 3000's cancellation fee?"
       found: "WidgetPro 3000's cancellation fee is $0."
       result: fee = $0  ✓

Final answer: "The cancellation fee is $0."
```

Both hops succeeded. The planner found everything it needed. But when the generator wrote the final answer, it dropped `WidgetPro 3000` — the identity established in hop 1 — and only carried the fee from hop 2.

The customer said thank you and went ahead with the migration. Two weeks later they were back, disputing whether that `$0` applied to their old product or their new one. The screenshot they showed read exactly: *"The cancellation fee is $0."* No product name. No way to verify.

**This is a memory failure.** The fact from hop 1 (`WidgetPro 3000`) did not survive into the final answer. A correct multi-hop answer requires both: the product identity from hop 1 AND the fee from hop 2. Losing the identity produces an answer the customer cannot act on with confidence.

**What the test must catch:** the final answer must name the product and give its fee. A response that gives only the fee — even the correct fee — is incomplete. The judge below evaluates this semantically: does the answer attribute the fee to a specific named product, or does it leave the attribution implicit?

### What the judge evaluates — Scenario A

The judge receives both the **retrieved contexts** from the agent's hop trace AND the **final answer**, then checks whether the key facts that were retrieved actually made it into the response:

```
Retrieved during hops:
  - "WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000."  ← hop 1 fact
  - "WidgetPro 3000's cancellation fee is $0."                                  ← hop 2 fact

Final answer: "The cancellation fee is $0."

Judge sees: "WidgetPro 3000" was in the retrieved contexts.
            "WidgetPro 3000" is not in the final answer.
            → retained=False — memory failure confirmed.
```

**Why the judge needs retrieved_contexts, not just domain knowledge:**
Without the hop trace, the judge can only ask "does the answer look complete?" — which is back to being a domain check, not a memory check. With the retrieved contexts, the judge can ask the actual question: *"which facts the agent found during retrieval failed to appear in what it said?"* That's memory validation. The failure is specifically that `WidgetPro 3000` crossed the retrieval boundary but didn't cross the generation boundary.

**Hard negative:**
```
retrieved_contexts:  ["WidgetPro 2000... replaced by WidgetPro 3000.", "WidgetPro 3000's fee is $0."]
response:            "The cancellation fee is $0."
verdict:             retained=False — the retrieved product identity didn't survive into the answer.
```

In [2]:
import json
from pydantic import BaseModel
from agent import agentic_rag, judge_client, judge_model

# Load all golden cases once — keyed by id, reused across all 3 checks below.
with open("golden_dataset.json") as f:
    _golden = {c["id"]: c for c in json.load(f)}


In [3]:
class MemoryRetentionVerdict(BaseModel):
    retained: bool
    reasoning: str


async def judge_memory_retention(
    question: str, retrieved_contexts: list[str], answer: str
) -> MemoryRetentionVerdict:
    contexts_block = "\n".join(f"- {c}" for c in retrieved_contexts)
    prompt = (
        "You are evaluating a support agent's answer for WidgetCo.\n\n"
        f"Customer question: {question}\n\n"
        "Facts the agent retrieved during its multi-hop search:\n"
        f"{contexts_block}\n\n"
        f"Agent's final answer: {answer}\n\n"
        "Check whether the key facts from the retrieved contexts survived into the final answer. "
        "A memory failure is when the agent retrieved a critical fact — such as the name of the "
        "replacement product — but that fact does not appear in the final answer, leaving the "
        "customer unable to verify the answer applies to their specific product.\n\n"
        "Set retained=true only if the final answer preserves the key entities and facts from "
        "the retrieved contexts that are necessary to make the answer complete and verifiable. "
        "A correct number with no product attribution is retained=false."
    )
    return await judge_client.chat.completions.create(
        model=judge_model,
        response_model=MemoryRetentionVerdict,
        messages=[{"role": "user", "content": prompt}],
    )


memory_case = _golden["multi-hop-widgetpro-01-memory-drop"]

# Real agent — live call. retrieved_contexts comes directly from the agent loop.
day1_result = await agentic_rag(memory_case["user_input"])
answer_with_memory = day1_result.response

# Hard negative: both facts were retrieved correctly, but the product identity was dropped.
# These are the contexts the correct 2-hop path would have produced.
hard_neg_contexts = [
    "WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000.",
    "WidgetPro 3000's cancellation fee is $0 -- it can be canceled anytime at no charge.",
]
answer_memory_dropped = memory_case["response"]  # "The cancellation fee is $0."

for label, answer, contexts in [
    ("REAL AGENT", answer_with_memory, day1_result.retrieved_contexts),
    ("HARD NEGATIVE (identity dropped)", answer_memory_dropped, hard_neg_contexts),
]:
    verdict = await judge_memory_retention(memory_case["user_input"], contexts, answer)
    print(f"[{label}]")
    print(f"  retrieved: {contexts}")
    print(f"  answer:    {answer!r}")
    print(f"  retained:  {verdict.retained}")
    print(f"  reasoning: {verdict.reasoning}")
    print()

print(f"num_hops={day1_result.num_hops}  hit_max_hops={day1_result.hit_max_hops}")


[REAL AGENT]
  retrieved: ["WidgetPro 2000's cancellation fee was $50 before it was discontinued.", "WidgetPro 3000's cancellation fee is $0 -- it can be canceled anytime at no charge.", 'WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000.']
  answer:    "The WidgetPro 2000 was replaced by the WidgetPro 3000. The facts state that the WidgetPro 3000's cancellation fee is $0."
  retained:  True
  reasoning: The final answer correctly identifies both the replacement product (WidgetPro 3000) and its cancellation fee ($0). It preserves the key entities necessary for the customer to verify that the answer applies specifically to the product that replaced the WidgetPro 2000. The information is complete and verifiable.

[HARD NEGATIVE (identity dropped)]
  retrieved: ['WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000.', "WidgetPro 3000's cancellation fee is $0 -- it can be canceled anytime at no charge."]
  answer:    'The cancellation fee is $0.'
  re

---

## Scenario B — "The Bait-and-Switch Fee"
### Incident 1 · Failure mode: wrong combination across hops

A customer who had just migrated from WidgetPro 2000 to WidgetPro 3000 asked the agent about canceling. The agent replied: *"The cancellation fee for the product that replaced WidgetPro 2000 is $50."*

Here is exactly what the agent did:

```
Hop 1  query: "What replaced WidgetPro 2000?"
       found: "WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000."
       result: replacement = WidgetPro 3000  ✓

Hop 2  query: "What is the cancellation fee?"
       found: "WidgetPro 2000's cancellation fee was $50 before it was discontinued."
       result: fee = $50  ✗  (retrieved the OLD product's fee, not the replacement's)

Final answer: "The cancellation fee for the product that replaced WidgetPro 2000 is $50."
```

Hop 1 is correct. Hop 2 retrieved the wrong product's fee — WidgetPro 2000's historical fee, not WidgetPro 3000's current one. The generator combined them: the replacement identity from hop 1 with the discontinued product's fee from hop 2.

The customer nearly paid it. WidgetPro 2000's fee really was `$50`. WidgetPro 3000, the replacement they'd migrated to, dropped that fee entirely — `$0`, cancel anytime. If they'd acted on the agent's answer, WidgetCo would owe them a refund.

**This is a reasoning failure.** Both retrieved facts are individually true and grounded. A faithfulness check looks at each fact in isolation — `$50` IS in the corpus, so it passes. But faithfulness cannot ask *which product's fee was the question asking about*. The combination is wrong even though each piece is right.

**What the test must catch:** did the agent apply WidgetPro 3000's fee (`$0`) or WidgetPro 2000's fee (`$50`) to the answer? The judge below is given both products and both fees, and reasons about which one was actually used — something a keyword check cannot do reliably.

### What the judge evaluates — Scenario B

The judge receives the actual retrieved contexts from both hops and the final answer, then checks whether the agent applied the right product's fee:

```
Retrieved during hops:
  - "WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000."  ← hop 1
  - "WidgetPro 2000's cancellation fee was $50 before it was discontinued."     ← hop 2 (wrong product's fee)

Final answer: "The cancellation fee for the product that replaced WidgetPro 2000 is $50."

Judge sees: hop 2 retrieved WidgetPro 2000's fee ($50), not WidgetPro 3000's.
            The answer applied that $50 to the replacement product.
            → correct_combination=False — the wrong product's fee was used.
```

**Why the judge needs retrieved_contexts here:**
The judge needs to see what was actually in the hop trace to determine *which* fee the agent had available and chose to apply. Without the hop trace, it can only check the answer against domain knowledge. With it, the judge can reason: "the agent had access to WidgetPro 2000's $50 fee in hop 2 — it used that one, but the question asked about the replacement."

**Hard negative (from golden dataset):**
```
retrieved_contexts:  hop 1 → replacement fact   hop 2 → WidgetPro 2000's $50 fee
response:            "The cancellation fee for the product that replaced WidgetPro 2000 is $50."
verdict:             correct_combination=False
```

In [ ]:
class ReasoningChainVerdict(BaseModel):
    correct_combination: bool
    reasoning: str


async def judge_reasoning_chain(
    question: str, retrieved_contexts: list[str], answer: str
) -> ReasoningChainVerdict:
    contexts_block = "\n".join(f"- {c}" for c in retrieved_contexts)
    prompt = (
        "You are evaluating a support agent's answer for WidgetCo.\n\n"
        f"Customer question: {question}\n\n"
        "Facts the agent retrieved during its multi-hop search:\n"
        f"{contexts_block}\n\n"
        f"Agent's final answer: {answer}\n\n"
        "Product context:\n"
        "- WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000.\n"
        "- WidgetPro 2000's cancellation fee was $50 before it was discontinued.\n"
        "- WidgetPro 3000's cancellation fee is $0 -- cancel anytime at no charge.\n\n"
        "The question asks for the cancellation fee of the REPLACEMENT product (WidgetPro 3000). "
        "Check whether the agent correctly used the replacement product's fee from the retrieved "
        "contexts, or mistakenly applied the discontinued product's fee ($50) instead. "
        "Both fees may appear in the retrieved contexts -- the error is which one was applied.\n\n"
        "Set correct_combination=true only if the answer correctly states that the replacement "
        "product (WidgetPro 3000) has a $0 cancellation fee."
    )
    return await judge_client.chat.completions.create(
        model=judge_model,
        response_model=ReasoningChainVerdict,
        messages=[{"role": "user", "content": prompt}],
    )


reasoning_neg = _golden["multi-hop-widgetpro-01-hardneg"]

for label, answer, contexts in [
    ("HARD NEGATIVE (wrong combination)", reasoning_neg["response"], reasoning_neg["retrieved_contexts"]),
    ("REAL AGENT (reusing run from Scenario A)", answer_with_memory, day1_result.retrieved_contexts),
]:
    verdict = await judge_reasoning_chain(reasoning_neg["user_input"], contexts, answer)
    print(f"[{label}]")
    print(f"  retrieved: {contexts}")
    print(f"  answer:             {answer!r}")
    print(f"  correct_combination: {verdict.correct_combination}")
    print(f"  reasoning:          {verdict.reasoning}")
    print()

print("The hard negative's retrieved facts are both individually true and grounded.")
print("RAGAS faithfulness would pass it. The judge catches it by seeing which fee")
print("from the hop trace was applied to the replacement product in the answer.")


---

## Scenario C — "The Phantom Policy"
### Incident 3 · Failure mode: ungraceful failure on an empty hop

A TurboMax Pro customer contacted support to dispute a charge of `$25` they said the AI agent had quoted them for canceling. The support team checked every system. WidgetCo has never had a `$25` cancellation fee for TurboMax Pro. The policy has never existed. The agent made it up.

Here is exactly what the agent did:

```
Hop 1  query: "What is the cancellation fee for TurboMax Pro?"
       found: "TurboMax Pro's annual subscription costs $299."  (price, not cancellation policy)
       planner: not enough — this is a price fact, not a fee policy

Hop 2  query: "TurboMax Pro cancellation fee policy"
       found: "TurboMax 5 was renamed to TurboMax Pro in 2024."  (rename history, not a policy)
       planner: not enough

Hop 3  query: "Cancellation fee for TurboMax Pro subscription"
       found: "TurboMax Pro's annual subscription costs $299."  (same subscription price, again)
       planner: not enough — exhausted max_hops
       hit_max_hops = True

Final answer: "The cancellation fee for TurboMax Pro is $25."
```

All three hops came up empty on the actual question. The corpus has TurboMax Pro's subscription price, the rename from TurboMax 5, and the general support plan. It has no cancellation policy for TurboMax Pro — it was simply never documented.

The planner correctly kept searching. It never confirmed `enough_info=True`. The agent's loop exhausted `max_hops` without finding anything, and `agent.py` told the generator explicitly: *"the planner was not confident these facts fully answer the question."* The generator should have hedged. Instead it produced `$25` — a number that appears nowhere in the corpus.

**This is a graceful-failure failure.** When all hops return irrelevant facts, the correct behavior is to admit the gap: *"I don't have cancellation policy information for TurboMax Pro."* Any specific number is wrong by definition — not because it's the wrong number, but because no number should have been given at all.

**What the test must catch:** did the agent admit it couldn't find the answer, or did it fabricate a specific figure? You cannot keyword-match all valid phrasings of "I don't know." That's exactly why this scenario needs an LLM judge — the judge can reason about intent, not just match strings.

### What the judge evaluates — Scenario C

The judge receives all three hops of retrieved contexts plus the final answer, then checks whether the agent admitted the knowledge gap or fabricated an answer from facts that don't cover the question:

```
Retrieved during hops:
  hop 1 → "TurboMax Pro's annual subscription costs $299."    (price, not a cancellation fee)
  hop 2 → "TurboMax 5 was renamed to TurboMax Pro in 2024."  (rename history, not a policy)
  hop 3 → "TurboMax Pro's annual subscription costs $299."    (same price fact, again)

Final answer: "The cancellation fee for TurboMax Pro is $25."

Judge sees: none of the retrieved contexts contain cancellation fee information.
            The answer produces a specific dollar amount not found in any retrieved context.
            → is_graceful=False — fabricated from a knowledge gap.
```

**Why the judge needs retrieved_contexts here:**
The judge needs the hop trace to confirm the knowledge gap actually exists — that the agent exhausted its search and genuinely found nothing relevant, rather than ignoring a fact that was there. With the full hop trace, the judge can verify: "all three hops returned subscription price or rename facts, never a cancellation policy — the fabrication is not a hallucination on top of a real answer, it's filling a genuine gap."

**Hard negative (from golden dataset):**
```
retrieved_contexts:  ["TurboMax Pro's annual subscription costs $299..."]
response:            "The cancellation fee for TurboMax Pro is $25."
verdict:             is_graceful=False — $25 appears nowhere in the retrieved contexts or corpus.
```

In [ ]:
class GracefulFailureVerdict(BaseModel):
    is_graceful: bool
    reasoning: str


async def judge_graceful_failure(
    question: str, retrieved_contexts: list[str], answer: str
) -> GracefulFailureVerdict:
    contexts_block = "\n".join(f"- {c}" for c in retrieved_contexts)
    prompt = (
        "You are evaluating a support agent's answer for WidgetCo.\n\n"
        f"Customer question: {question}\n\n"
        "Facts the agent retrieved during its multi-hop search:\n"
        f"{contexts_block}\n\n"
        f"Agent's final answer: {answer}\n\n"
        "None of the retrieved contexts contain information about TurboMax Pro's cancellation fee. "
        "When a retrieval loop finds no relevant facts, the correct behavior is to admit the gap -- "
        "any phrasing that communicates 'I don't have that information' is acceptable. "
        "Fabricating a specific figure (a dollar amount, date, or policy) that does not appear "
        "in any of the retrieved contexts is ungraceful failure.\n\n"
        "Set is_graceful=true only if the answer explicitly admits the knowledge gap. "
        "Set is_graceful=false if the answer produces a specific fact not supported by the "
        "retrieved contexts above."
    )
    return await judge_client.chat.completions.create(
        model=judge_model,
        response_model=GracefulFailureVerdict,
        messages=[{"role": "user", "content": prompt}],
    )


graceful_neg = _golden["graceful-failure-01-hardneg"]

# Real agent on a question the corpus cannot answer — all hops return irrelevant facts.
missing_fact_result = await agentic_rag(graceful_neg["user_input"], verbose=True)
real_agent_answer = missing_fact_result.response

for label, answer, contexts in [
    ("REAL AGENT", real_agent_answer, missing_fact_result.retrieved_contexts),
    ("HARD NEGATIVE (fabricated)", graceful_neg["response"], graceful_neg["retrieved_contexts"]),
]:
    verdict = await judge_graceful_failure(graceful_neg["user_input"], contexts, answer)
    print(f"[{label}]")
    print(f"  retrieved: {contexts}")
    print(f"  answer:     {answer!r}")
    print(f"  is_graceful: {verdict.is_graceful}")
    print(f"  reasoning:  {verdict.reasoning}")
    print()

print(f"missing_fact_result.hit_max_hops = {missing_fact_result.hit_max_hops}")
print("True = the planner never confirmed enough_info — confirmed the knowledge gap exists.")
print("If this were False, the planner hallucinated confidence — a separate finding worth logging.")


---
## Extending the coverage matrix

Module 5 Day 2 added retrieval columns to Module 4 Day 4's matrix. Today adds the agentic ones -- and they map directly onto this notebook's opening argument: `reasoning_chain_break` is the check for the failure faithfulness can't see, `ungraceful_failure` is the check for the failure no RAGAS metric asks about at all.


In [6]:
import json
from collections import Counter

with open("golden_dataset.json") as f:
    golden_cases = json.load(f)

categories    = sorted({c["category"] for c in golden_cases})
failure_modes = sorted({c["failure_mode"] for c in golden_cases})
counts        = Counter((c["category"], c["failure_mode"]) for c in golden_cases)

header = " " * 16 + "".join(f"{fm:<24}" for fm in failure_modes)
print(header)
for cat in categories:
    row = f"{cat:<16}" + "".join(f"{counts[(cat, fm)]:<24}" for fm in failure_modes)
    print(row)

zero_cells = [(cat, fm) for cat in categories for fm in failure_modes if counts[(cat, fm)] == 0]
print()
if zero_cells:
    print("Still-empty cells (not necessarily a problem -- just visible now):")
    for cat, fm in zero_cells:
        print(f"- {cat} x {fm}")
else:
    print("No empty cells for these categories x failure modes -- Day 1's named premature_stop gap")
    print("is filled (see multi-hop-turbomax-01 in golden_dataset.json). single_hop_qa correctly")
    print("has no agentic-failure rows -- those columns only apply once a question needs multiple hops.")


                hallucination           premature_stop          reasoning_chain_break   ungraceful_failure      
multi_hop_qa    2                       2                       3                       2                       
single_hop_qa   2                       0                       0                       0                       

Still-empty cells (not necessarily a problem -- just visible now):
- single_hop_qa x premature_stop
- single_hop_qa x reasoning_chain_break
- single_hop_qa x ungraceful_failure


---
## Summary — What you built as a QA engineer today

You started with three real support incidents. You ended with a reproducible test suite that would catch all three before the next release.

### What you built
| Scenario | Incident | Failure mode | Test mechanism |
|---|---|---|---|
| A — The Memory Drop | Customer got `$0` for an unnamed product | `reasoning_chain_break` (memory) | `must_include: ["WidgetPro 3000"]` + LLM judge |
| B — The Bait-and-Switch Fee | Customer was quoted `$50` on a `$0` product | `reasoning_chain_break` (wrong combination) | `must_include` + `must_not_include` + LLM judge |
| C — The Phantom Policy | Customer was quoted `$25` that doesn't exist | `ungraceful_failure` | `hit_max_hops` check + LLM judge |

### The thread back through the course
Every technique here was something you'd already seen — pointed one layer higher:

- **Equivalence partitioning (Module 4 Day 4)** → applied to hop count and failure modes, not just input ranges
- **Boundary value analysis (Module 5 Day 2)** → the hop boundary is the same kind of edge as a chunk boundary
- **Hard negatives (Module 4 Day 4 + Module 5 Day 3)** → right facts, wrong combination; grounded but fabricated
- **Coverage matrix (Module 4 Day 4)** → new columns for agentic failure modes, same habit
- **LangSmith tracing (Module 5)** → now load-bearing: without per-hop spans you can't tell Scenario A from Scenario B from a clean success in the logs

### What changed from Module 5
Module 5's RAGAS suite is still correct — and completely insufficient for these three incidents. Not because it's weak but because it was built for a different system. A faithfulness check on a single retrieval can't see a combination error across two hops. Answer relevancy can't see a memory drop. Neither can see what *should have* been retrieved and wasn't.

The right mental model: RAGAS tests the output. The three judges today test the *reasoning process* that produced it. Both layers are necessary — neither is enough alone.

### What WidgetCo gets
Before today: 500 queries a day, no systematic test coverage, three real incidents already logged.
After today: three targeted test cases, each with a hard negative that would have caught the real incident, run against a live agent, with reproducible LLM-judge verdicts that update automatically on every new model version.

**Next:** Module 7 — AI Agents Testing with DeepEval, where `judge_memory_retention`, `judge_reasoning_chain`, and `judge_graceful_failure` become formal, reusable metrics in a proper framework, and the agent gains real tool-calling instead of just retrieval.

---